# Assignment 8: REST API and Networking Lab

**BLCH9X2 Blockchain, Master of Financial Engineering, University of Johannesburg**

Bukho Ndziweni, student number 220081696

A blockchain node with a REST (Representational State Transfer) API (Application Programming Interface), followed by one payment travelling from a client's request to a confirmed block and then across three nodes.

| Section | Contents | Brief item |
|---|---|---|
| 1 | Setup | |
| 2 to 5 | The four code files, saved to disk by the notebook | A (a), A (c) |
| 6 | Three local nodes are started | A (a) |
| 7 | A full payment on one node | A (b) |
| 8 | The explorer | A (c) |
| 9 | Three nodes: address relay, `inv` and `getdata` | Part B |
| 10 | Checks | |

**To run:** `pip install flask requests`, then *Restart Kernel and Run All Cells*. It takes a few seconds. Every hash, nonce and balance is the same on any machine. Only the mining seconds differ.

## 1. Setup

The two external libraries are Flask, which serves the API, and requests, which sends requests to it.

In [1]:
import importlib                           # importlib loads a code file, and reloads it after it has been rewritten
import json                                # json prints replies in an indented, readable form
import sys                                 # sys reports which version of Python is running
from importlib.metadata import version     # version() reads the installed version of a library

from IPython.display import HTML, display  # shows a web page inside the notebook

print("Python  ", sys.version.split()[0])  # the Python version, for example 3.11.5
print("Flask   ", version("flask"))        # the Flask version
print("requests", version("requests"))     # the requests version

Python   3.13.9
Flask    3.1.2
requests 2.32.5


## 2. The chain core: `blockchain.py`

Blocks, mining, the coinbase reward and the pending pool, carried forward from Assignments 2 to 6. New here: every transaction has a `txid`, and a payment must be affordable to be accepted.

The first line, `%%writefile`, saves the cell to a file instead of running it. Sections 2 to 5 therefore produce the four `.py` files of the submission.

In [2]:
%%writefile blockchain.py
"""
blockchain.py
Assignment 8: REST API and Networking Lab (the chain core)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS FILE DOES
    Holds the ledger that the Application Programming Interface (API) in api.py exposes.
    It carries forward the semester build: blocks sealed with SHA-256 (Secure Hash
    Algorithm, 256-bit), proof-of-work mining, a coinbase reward and a pending pool.

        Block        one batch of transactions, sealed with a hash
        Blockchain   the ordered list of blocks plus the pool of pending payments

    Two rules are new in this assignment, because a payment API needs them:

        1. A payment is only accepted if the sender can afford it, counting payments
           the sender already has waiting in the pending pool.
        2. Every transaction has an identifier (txid), the hash of its own fields.
           Submitting the same payment twice gives the same txid and is refused, so a
           retried request can never pay twice.

HOW TO RUN
    This file is a library and is not run directly. Start a node with:
        python api.py

REPRODUCIBILITY
    Block timestamps come from a logical clock (genesis time plus ten minutes per
    block) and every nonce search starts at zero, so every nonce, hash and txid is
    identical on any machine. Only the mining seconds differ.
"""

import hashlib          # hashlib gives us SHA-256, the fingerprint function
import json             # json turns Python dictionaries into text in a controlled, repeatable way
import re               # re lets us check that an address only uses permitted characters
import time             # time lets us measure how long mining takes, in seconds

# ===========================================================================
# CONFIGURATION
# ===========================================================================

DIFFICULTY = 4                        # our mining rule: a valid block hash must start with this many zero characters
GENESIS_TIMESTAMP = 1790812800        # a fixed Unix timestamp (01 October 2026, 00:00:00 UTC, Coordinated Universal Time)
SECONDS_PER_BLOCK = 600               # the logical clock moves 600 seconds (10 minutes) for every block, as Bitcoin aims for
GENESIS_PREVIOUS_HASH = "0" * 64      # 64 zeros, the agreed convention meaning "there is no earlier block"
BLOCK_SUBSIDY = 50                    # new units created by the rules and paid to whoever mines a block
FEE_PER_PAYMENT = 1                   # a flat charge the sender pays to the miner for including a payment
MAX_TEXT_LENGTH = 64                  # the longest address or payment reference the node will accept
ADDRESS_PATTERN = re.compile(r"[A-Za-z0-9_-]+")   # an address may use letters, digits, underscores and hyphens only

# ===========================================================================
# HASHING HELPERS
# ===========================================================================

def canonical_json(payload) -> str:
    """Turn a dictionary into one single agreed text string, so hashing is repeatable."""
    # sort_keys=True    : always writes the fields in alphabetical order, so field order can never change the hash
    # separators=(...)  : removes all optional spaces, so spacing can never change the hash
    # ensure_ascii=True : escapes any non-English characters, so the text is identical on every machine
    return json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=True)


def sha256_hex(text: str) -> str:
    """Return the SHA-256 fingerprint of a piece of text, written as 64 hexadecimal characters."""
    encoded_text = text.encode("utf-8")           # convert the text into raw bytes, because hashing works on bytes, not letters
    digest = hashlib.sha256(encoded_text)         # run the SHA-256 algorithm over those bytes
    return digest.hexdigest()                     # return the result as a readable 64 character string


def meets_difficulty(hash_hex: str, difficulty: int = DIFFICULTY) -> bool:
    """Return True only if the hash starts with the required number of zeros (our proof-of-work target)."""
    required_prefix = "0" * difficulty            # build the pattern we demand, for example "0000" when difficulty is 4
    return hash_hex.startswith(required_prefix)   # True means this hash was mined properly, False means it was not


def block_timestamp(index: int) -> int:
    """Return the timestamp the logical clock assigns to the block at this position."""
    return GENESIS_TIMESTAMP + index * SECONDS_PER_BLOCK   # genesis time plus ten minutes for every block after it

# ===========================================================================
# TRANSACTIONS
# ===========================================================================

def is_address(value) -> bool:
    """Return True if the value is usable as an account address."""
    if not isinstance(value, str):                          # an address has to be text, not a number or a list
        return False                                        # anything else is refused
    if len(value) > MAX_TEXT_LENGTH:                        # very long names are refused to keep records small
        return False                                        # too long, so not a valid address
    return ADDRESS_PATTERN.fullmatch(value) is not None     # True only if every character is a permitted one


def transaction_id(fields: dict) -> str:
    """Return the identifier of a transaction: the SHA-256 hash of every field except the identifier itself."""
    body = {name: value for name, value in fields.items() if name != "txid"}   # copy the fields, leaving out the txid
    return sha256_hex(canonical_json(body))                                    # fingerprint the agreed text form of those fields


def make_payment(sender: str, receiver: str, amount: int, reference: str = "") -> dict:
    """Build one ordinary payment, which pays a flat fee to whichever miner includes it."""
    payment = {                               # a dictionary holding the fields of one payment
        "type": "payment",                    # marks this as an ordinary transfer rather than a reward
        "sender": sender,                     # who is paying
        "receiver": receiver,                 # who is being paid
        "amount": amount,                     # the value, in whole units so no rounding can ever occur
        "fee": FEE_PER_PAYMENT,               # what the sender pays the miner for including this payment
        "reference": reference,               # the payer's own reference, which also keeps repeat payments distinct
    }
    payment["txid"] = transaction_id(payment)   # the identifier is the hash of all the fields above
    return payment                              # hand back the finished payment


def make_coinbase(miner_address: str, block_index: int, total_fees: int) -> dict:
    """Build the special first transaction that pays the miner for finding this block."""
    coinbase = {                                            # the coinbase has no sender: these units are newly created
        "type": "coinbase",                                 # marks this as the reward rather than a transfer
        "sender": None,                                     # nobody is debited, so the sender is left empty
        "receiver": miner_address,                          # the address being paid
        "amount": BLOCK_SUBSIDY + total_fees,               # the fixed subsidy plus every fee collected in this block
        "fee": 0,                                           # a reward pays no fee
        "reference": f"reward for block {block_index}",     # naming the block stops the same coinbase being reused elsewhere
    }
    coinbase["txid"] = transaction_id(coinbase)   # the identifier is the hash of all the fields above
    return coinbase                               # hand back the finished reward transaction


def payment_field_error(data):
    """Check the fields of a submitted payment. Return a message describing the first problem, or None if all is well."""
    if not isinstance(data, dict):                                              # the submission must be a set of named fields
        return "the request body must be a JSON object"                         # JSON is JavaScript Object Notation
    for field in ("sender", "receiver"):                                        # both parties are checked the same way
        if not is_address(data.get(field)):                                     # .get returns None when the field is missing
            return f"'{field}' must be 1 to {MAX_TEXT_LENGTH} letters, digits, hyphens or underscores"   # explain the rule that failed
    if data["sender"] == data["receiver"]:                                      # paying yourself moves nothing
        return "'sender' and 'receiver' must be different"                      # so it is refused
    amount = data.get("amount")                                                 # read the amount, or None if it is missing
    if isinstance(amount, bool) or not isinstance(amount, int) or amount <= 0:  # Python treats True as 1, so booleans are excluded first
        return "'amount' must be a whole number greater than zero"              # decimals, text and negatives are all refused
    reference = data.get("reference", "")                                       # the reference is optional and defaults to empty
    if not isinstance(reference, str) or len(reference) > MAX_TEXT_LENGTH:      # it must be text and not too long
        return f"'reference' must be text of at most {MAX_TEXT_LENGTH} characters"   # explain the rule that failed
    return None                                                                 # None means no problem was found


def apply_transaction(balances: dict, transaction: dict):
    """Move the value of one transaction through a dictionary of balances."""
    if transaction["type"] == "payment":                                                  # only ordinary payments debit somebody
        cost = transaction["amount"] + transaction["fee"]                                 # the sender pays the amount plus the fee
        balances[transaction["sender"]] = balances.get(transaction["sender"], 0) - cost   # debit the sender
    receiver = transaction["receiver"]                                                    # whoever is being paid
    balances[receiver] = balances.get(receiver, 0) + transaction["amount"]                # credit the receiver (or the miner, for a coinbase)

# ===========================================================================
# THE BLOCK CLASS, CARRIED FORWARD FROM ASSIGNMENT 2
# ===========================================================================

class Block:
    """One block in the chain: a batch of transactions, sealed with a fingerprint."""

    def __init__(self, index: int, timestamp: int, transactions: list, previous_hash: str, nonce: int = 0):
        """Create a block and immediately seal it by computing its hash."""
        self.index = index                     # the position of this block in the chain, starting at 0 for genesis
        self.timestamp = timestamp             # when the block was created, as a Unix timestamp (whole seconds)
        self.transactions = transactions       # the list of transactions this block carries
        self.previous_hash = previous_hash     # the fingerprint of the block before this one, which is what forms the chain
        self.nonce = nonce                     # the "number used once", the only field a miner is allowed to change freely
        self.hash = self.compute_hash()        # seal the block by fingerprinting everything above

    def header_payload(self) -> dict:
        """Collect every field the hash must commit to, leaving out the hash itself."""
        return {                                 # note that self.hash is deliberately NOT included, or we would hash the answer
            "index": self.index,                 # committing to position stops a block being moved elsewhere in the chain
            "timestamp": self.timestamp,         # committing to time stops the block being back-dated
            "transactions": self.transactions,   # committing to the payments is what makes the amounts tamper evident
            "previous_hash": self.previous_hash, # committing to the parent is what links the chain together
            "nonce": self.nonce,                 # committing to the nonce is what makes the mining work count
        }

    def compute_hash(self) -> str:
        """Recompute this block's fingerprint from its current contents."""
        serialised = canonical_json(self.header_payload())  # write the fields as one agreed text string
        return sha256_hex(serialised)                       # fingerprint that text with SHA-256

    def to_dict(self) -> dict:
        """Return the block as a plain dictionary, the form in which it travels over the network."""
        as_dict = self.header_payload()   # start from the five fields the hash commits to
        as_dict["hash"] = self.hash       # then add the stored hash so the reader can check it
        return as_dict                    # hand back all six fields

    @staticmethod
    def from_dict(data):
        """Rebuild a block from a dictionary received from another node. Return None if the data is unusable."""
        try:                                                    # data from another node cannot be trusted to have the right shape
            block = Block(                                      # build the block from the five hashed fields
                index=int(data["index"]),                       # the claimed position, forced to a whole number
                timestamp=int(data["timestamp"]),               # the claimed time, forced to a whole number
                transactions=list(data["transactions"]),        # the claimed transactions, forced to a list
                previous_hash=str(data["previous_hash"]),       # the claimed parent hash, forced to text
                nonce=int(data["nonce"]),                       # the claimed nonce, forced to a whole number
            )
            block.hash = str(data["hash"])                      # keep the hash the sender CLAIMED, so it can be checked later
            return block                                        # hand back the rebuilt block
        except (KeyError, TypeError, ValueError):               # a missing field or a wrong type lands here
            return None                                         # None tells the caller the data was unusable

# ===========================================================================
# MINING AND THE GENESIS BLOCK
# ===========================================================================

def mine_block(block: Block, difficulty: int = DIFFICULTY) -> tuple:
    """Try nonce values one after another until the block hash meets the difficulty rule."""
    started = time.perf_counter()                 # start a stopwatch so the cost of mining can be reported
    block.nonce = 0                               # every search starts at zero, which makes the result repeatable
    block.hash = block.compute_hash()             # hash the block with that first nonce
    attempts = 1                                  # one hash has been tried so far
    while not meets_difficulty(block.hash, difficulty):   # keep going until the hash starts with enough zeros
        block.nonce += 1                          # move to the next nonce
        block.hash = block.compute_hash()         # hash the block again with the new nonce
        attempts += 1                             # count this attempt
    seconds = time.perf_counter() - started       # stop the stopwatch
    return attempts, seconds                      # report how many hashes it took and how long


_GENESIS_FIELDS = None   # the mined genesis block is remembered here so it only has to be mined once per program


def make_genesis_block() -> Block:
    """Return block 0, the only block that has no parent. Every node builds exactly the same one."""
    global _GENESIS_FIELDS                                # allow this function to store the remembered genesis block
    if _GENESIS_FIELDS is None:                           # the first call has to do the mining
        genesis = Block(                                  # build the block using fixed values only
            index=0,                                      # genesis always sits at position 0
            timestamp=GENESIS_TIMESTAMP,                  # the fixed starting time from the configuration
            transactions=[],                              # genesis carries no transactions, so every balance starts at zero
            previous_hash=GENESIS_PREVIOUS_HASH,          # 64 zeros, because there is no earlier block
        )
        mine_block(genesis)                               # genesis obeys the same difficulty rule as every other block
        _GENESIS_FIELDS = genesis.to_dict()               # remember the result for later calls
    return Block.from_dict(_GENESIS_FIELDS)               # hand each caller its own separate copy

# ===========================================================================
# VALIDATION RULES SHARED BY EVERY NODE
# ===========================================================================

def block_error(block: Block, previous_block: Block, balances: dict, seen_txids: set):
    """Check one block against the block before it. Return a message describing the first problem, or None.

    When the block is valid, balances and seen_txids are updated to include it."""
    if block.index != previous_block.index + 1:                         # rule 1: positions must count up by one
        return "index does not follow the previous block"               # the block is out of sequence
    if block.previous_hash != previous_block.hash:                      # rule 2: the block must point at its parent
        return "previous_hash does not match the previous block"        # the link is broken
    if block.timestamp != block_timestamp(block.index):                 # rule 3: the time must follow the logical clock
        return "timestamp does not follow the logical clock"            # the block claims a time the rules do not allow
    if block.hash != block.compute_hash():                              # rule 4: the stored hash must match the contents
        return "stored hash does not match the block contents"          # something was edited after sealing
    if not meets_difficulty(block.hash):                                # rule 5: the hash must show proof of work
        return "hash does not meet the proof-of-work difficulty"        # the block was not mined
    transactions = block.transactions                                   # rules 6 to 8 concern the transactions inside
    if not transactions or not all(isinstance(item, dict) for item in transactions):   # there must be at least a coinbase, all as records
        return "block has no usable transactions"                       # an empty or malformed list is refused
    reward, payments = transactions[0], transactions[1:]                # the first transaction is the reward, the rest are payments
    if not is_address(reward.get("receiver")):                          # the reward must be paid to a usable address
        return "coinbase receiver is not a valid address"               # refuse a reward with a broken receiver
    expected_reward = make_coinbase(reward["receiver"], block.index, FEE_PER_PAYMENT * len(payments))   # rebuild the only reward the rules allow
    if reward != expected_reward:                                       # rule 6: the reward must be exactly subsidy plus fees
        return "coinbase is not the reward the rules allow"             # a miner tried to overpay itself or reuse a reward
    working_balances = dict(balances)                                   # work on a copy, so a rejected block changes nothing
    working_txids = set(seen_txids)                                     # likewise for the set of identifiers already used
    apply_transaction(working_balances, reward)                         # credit the miner
    working_txids.add(reward["txid"])                                   # remember the reward's identifier
    for payment in payments:                                            # now check every ordinary payment in order
        problem = payment_field_error(payment)                          # are the fields well formed?
        if problem is not None:                                         # a malformed payment invalidates the block
            return f"payment is malformed: {problem}"                   # report which rule failed
        rebuilt = make_payment(payment["sender"], payment["receiver"], payment["amount"], payment.get("reference", ""))   # rebuild it from its own fields
        if payment != rebuilt:                                          # rule 7: the txid and fee must match the fields
            return "payment does not match its txid"                    # the payment was altered after it was created
        if payment["txid"] in working_txids:                            # rule 7 continued: no payment may appear twice
            return "payment appears more than once in the chain"        # a replayed payment is refused
        cost = payment["amount"] + payment["fee"]                       # what the sender must be able to afford
        if working_balances.get(payment["sender"], 0) < cost:           # rule 8: no sender may spend more than it holds
            return "payment spends more than the sender holds"          # an overdrawn payment invalidates the block
        apply_transaction(working_balances, payment)                    # move the value
        working_txids.add(payment["txid"])                              # remember the identifier
    balances.clear()                                                    # the block is valid, so publish the new balances
    balances.update(working_balances)                                   # by copying the working values back
    seen_txids.update(working_txids)                                    # and publish the identifiers it used
    return None                                                         # None means the block passed every rule


def chain_error(blocks: list):
    """Check a whole chain from genesis. Return a message describing the first problem, or None."""
    if not blocks or any(block is None for block in blocks):            # an empty chain or an unreadable block is refused
        return "chain is empty or contains an unreadable block"         # nothing further can be checked
    if blocks[0].to_dict() != make_genesis_block().to_dict():           # every node must start from the same genesis block
        return "chain does not start from the shared genesis block"     # this chain belongs to a different network
    balances, seen_txids = {}, set()                                    # replay the chain from nothing
    for previous_block, block in zip(blocks, blocks[1:]):               # walk each block together with the one before it
        problem = block_error(block, previous_block, balances, seen_txids)   # apply the block rules
        if problem is not None:                                         # stop at the first broken block
            return f"block {block.index}: {problem}"                    # say which block failed and why
    return None                                                         # None means the whole chain is valid

# ===========================================================================
# THE BLOCKCHAIN CLASS: THE CHAIN PLUS THE PENDING POOL
# ===========================================================================

class Blockchain:
    """The ledger held by one node: its chain of blocks and its pool of payments waiting to be mined."""

    def __init__(self):
        """Start a new ledger holding only the genesis block."""
        self.chain = [make_genesis_block()]   # the ordered list of blocks, oldest first
        self.pending = []                     # payments accepted but not yet mined, oldest first

    @property
    def last_block(self) -> Block:
        """Return the newest block, which the next block must point at."""
        return self.chain[-1]                 # "[-1]" means the last item in a list

    # -----------------------------------------------------------------------
    # Reading the ledger
    # -----------------------------------------------------------------------
    def confirmed_balances(self) -> dict:
        """Replay every mined transaction and return what each address holds."""
        balances = {}                                 # start with nobody holding anything
        for block in self.chain:                      # walk the chain from genesis
            for transaction in block.transactions:    # and every transaction inside each block
                apply_transaction(balances, transaction)   # move its value
        return balances                               # the finished table of address to balance

    def confirmed_txids(self) -> set:
        """Return the identifier of every transaction already mined into the chain."""
        return {transaction["txid"] for block in self.chain for transaction in block.transactions}   # one pass over every block

    def balance_of(self, address: str) -> dict:
        """Report what an address holds, what it has waiting to leave, and what it can still spend."""
        confirmed = self.confirmed_balances().get(address, 0)      # the mined balance, or zero for an unknown address
        pending_out = sum(payment["amount"] + payment["fee"]       # add up amount plus fee
                          for payment in self.pending              # over every payment in the pool
                          if payment["sender"] == address)         # that this address is sending
        return {                                                   # a small report in three parts
            "address": address,                                    # whose balance this is
            "confirmed": confirmed,                                # value already mined into the chain
            "pending_out": pending_out,                            # value committed to payments still in the pool
            "available": confirmed - pending_out,                  # what the address may still spend
        }

    def find_transaction(self, txid: str):
        """Look a transaction up by identifier. Return its status, or None if this node has never seen it."""
        for block in self.chain:                                   # search the mined blocks first
            for transaction in block.transactions:                 # and every transaction inside each
                if transaction["txid"] == txid:                    # found it in the chain
                    return {                                       # report where it was mined
                        "status": "confirmed",                     # it is part of the chain
                        "block_index": block.index,                # the block that holds it
                        "confirmations": self.last_block.index - block.index + 1,   # that block plus every block built on top
                        "transaction": transaction,                # the transaction itself
                    }
        for payment in self.pending:                               # otherwise search the pending pool
            if payment["txid"] == txid:                            # found it waiting
                return {                                           # report that it is not yet mined
                    "status": "pending",                           # it is waiting for a block
                    "block_index": None,                           # so it has no block yet
                    "confirmations": 0,                            # and no confirmations
                    "transaction": payment,                        # the payment itself
                }
        return None                                                # this node has never seen that identifier

    def get_block(self, block_hash: str):
        """Return the block with this hash, or None if this node does not hold it."""
        for block in self.chain:              # search the chain
            if block.hash == block_hash:      # compare each stored hash with the one asked for
                return block                  # found it
        return None                           # no block with that hash

    # -----------------------------------------------------------------------
    # Changing the ledger
    # -----------------------------------------------------------------------
    def add_payment(self, data) -> tuple:
        """Try to place a payment in the pending pool. Return (outcome, message, payment)."""
        problem = payment_field_error(data)                                  # first check the shape of the fields
        if problem is not None:                                              # a malformed payment goes no further
            return "invalid", problem, None                                  # say what was wrong
        payment = make_payment(data["sender"], data["receiver"], data["amount"], data.get("reference", ""))   # build the payment and its txid
        if "txid" in data and data != payment:                               # a payment relayed by another node arrives with a txid
            return "invalid", "payment does not match its txid", None        # refuse it if the fields were altered on the way
        already_pending = any(item["txid"] == payment["txid"] for item in self.pending)   # is the same payment already waiting?
        if already_pending or payment["txid"] in self.confirmed_txids():     # or has it already been mined?
            return "duplicate", "this exact payment has already been submitted; change 'reference' to pay again", payment   # refuse the repeat
        cost = payment["amount"] + payment["fee"]                            # what this payment takes from the sender
        available = self.balance_of(payment["sender"])["available"]          # what the sender can still spend
        if cost > available:                                                 # the affordability rule
            return "insufficient_funds", f"'{payment['sender']}' needs {cost} (amount plus fee) but has {available} available", payment   # refuse it
        self.pending.append(payment)                                         # all checks passed, so the payment joins the pool
        return "accepted", "payment accepted into the pending pool", payment # report success

    def mine_pending(self, miner_address: str) -> tuple:
        """Mine one block holding the reward and every pending payment. Return (block, attempts, seconds)."""
        index = self.last_block.index + 1                                    # the new block takes the next position
        total_fees = FEE_PER_PAYMENT * len(self.pending)                     # every pending payment contributes one fee
        coinbase = make_coinbase(miner_address, index, total_fees)           # the reward for whoever is mining
        block = Block(                                                       # assemble the candidate block
            index=index,                                                     # its position
            timestamp=block_timestamp(index),                                # its time on the logical clock
            transactions=[coinbase] + self.pending,                          # the reward first, then the payments in arrival order
            previous_hash=self.last_block.hash,                              # the link to the current newest block
        )
        attempts, seconds = mine_block(block)                                # the proof-of-work search, the only slow step
        self.chain.append(block)                                             # the mined block joins the chain
        self.pending = []                                                    # its payments are no longer pending
        return block, attempts, seconds                                      # report the block and what it cost

    def accept_block(self, block: Block) -> tuple:
        """Try to add a block received from another node. Return (outcome, message)."""
        if self.get_block(block.hash) is not None:                           # this node already holds a block with that hash
            return "duplicate", "block already held"                         # nothing to do
        if block.previous_hash != self.last_block.hash:                      # the block builds on something other than our newest block
            return "detached", "block does not extend this node's newest block"   # the caller will compare whole chains instead
        problem = block_error(block, self.last_block, self.confirmed_balances(), self.confirmed_txids())   # apply every block rule
        if problem is not None:                                              # an invalid block is never stored
            return "rejected", problem                                       # say which rule it broke
        self.chain.append(block)                                             # the block is valid, so it joins the chain
        self._rebuild_pending(self.pending)                                  # payments it mined leave the pending pool
        return "accepted", "block appended"                                  # report success

    def replace_chain(self, blocks: list) -> tuple:
        """Adopt another node's chain if it is valid and longer than ours. Return (replaced, message)."""
        if len(blocks) <= len(self.chain):                                   # only a strictly longer chain can win
            return False, "the other chain is not longer"                    # ties keep the chain seen first
        problem = chain_error(blocks)                                        # validate the candidate from genesis
        if problem is not None:                                              # an invalid chain never wins, whatever its length
            return False, f"the other chain is invalid ({problem})"          # say why it was refused
        dropped = [transaction                                               # collect the payments
                   for block in self.chain                                   # from every block of the chain being abandoned
                   for transaction in block.transactions                     # looking at each transaction
                   if transaction["type"] == "payment"]                      # rewards from abandoned blocks simply vanish
        self.chain = blocks                                                  # adopt the longer chain
        self._rebuild_pending(dropped + self.pending)                        # payments the new chain lacks go back to the pool
        return True, "adopted the longer valid chain"                        # report the switch

    def _rebuild_pending(self, candidates: list):
        """Refill the pending pool, keeping only payments that are still unmined and still affordable."""
        self.pending = []                     # start from an empty pool
        for payment in candidates:            # offer each candidate back to the pool in its original order
            self.add_payment(payment)         # the normal checks drop anything already mined or no longer affordable

Overwriting blockchain.py


## 3. The peer-to-peer layer: `network.py`

The peer list and the gossip rules. Each Bitcoin message has one HTTP (HyperText Transfer Protocol) counterpart, listed at the top of the file.

In [3]:
%%writefile network.py
"""
network.py
Assignment 8: REST API and Networking Lab (the peer-to-peer layer)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS FILE DOES
    Gives each node a list of peers and a way to gossip with them. Bitcoin nodes talk
    over long-lived connections using named messages. This lab keeps the same ideas
    but sends each message as one HTTP (HyperText Transfer Protocol) request:

        Bitcoin message         This lab
        ---------------------   --------------------------------------------------
        version / verack        the "from" field of POST /nodes/register
        getaddr                 GET  /nodes
        addr                    POST /nodes/register   (body: the addresses)
        inv                     POST /inv              (body: a type and a hash)
        getdata                 GET  /getdata/<type>/<hash>
        tx / block              the JSON (JavaScript Object Notation) reply to getdata
        getheaders, getblocks   GET  /chain            (used by GET /nodes/resolve)

    PeerBook   the peer list, restricted to this computer
    Node       one participant: a Blockchain, a PeerBook and the gossip rules

    Gossip stops by itself because a node only passes on what was new to it: an
    address it did not know, or a hash it did not hold.

SAFETY
    Only loopback addresses (127.0.0.1 or localhost) can be registered as peers, so a
    node can never be told to send requests to another computer.

HOW TO RUN
    This file is a library and is not run directly. Start a node with:
        python api.py
"""

import itertools                    # itertools.count gives us a simple counter for numbering log entries
import threading                    # threading provides locks, so two requests cannot change the ledger at the same moment
from urllib.parse import urlsplit   # urlsplit breaks a web address into its host and port

import requests                     # requests sends HTTP requests to other nodes

from blockchain import DIFFICULTY, Block, Blockchain, is_address   # the chain core this layer sits on top of

# ===========================================================================
# CONFIGURATION
# ===========================================================================

LOOPBACK_HOSTS = ("127.0.0.1", "localhost")   # the only hosts a peer may live on: both names mean "this computer"
MAX_PEERS = 16                                # an upper limit on the peer list, so it cannot grow without bound
REQUEST_TIMEOUT = 10                          # seconds to wait for another node before giving up
MAX_EVENTS = 500                              # how many log entries each node remembers
EVENT_COUNTER = itertools.count(1)            # one counter shared by every node in this program, starting at 1

# ===========================================================================
# ADDRESS AND REQUEST HELPERS
# ===========================================================================

def normalise_address(text):
    """Turn "http://localhost:5001/" or "127.0.0.1:5001" into "127.0.0.1:5001". Return None if it is not a local node."""
    if not isinstance(text, str) or not text.strip():          # a peer address has to be non-empty text
        return None                                            # anything else is unusable
    candidate = text.strip()                                   # remove any spaces around the address
    if "//" not in candidate:                                  # urlsplit only finds the host when a scheme is present
        candidate = "http://" + candidate                      # so add one when the caller left it out
    try:                                                       # a malformed address can make the parser raise an error
        parts = urlsplit(candidate)                            # split the address into scheme, host, port and path
        host, port = parts.hostname, parts.port                # pick out the two parts we need
    except ValueError:                                         # for example a port that is not a number
        return None                                            # unusable
    if parts.scheme != "http" or host not in LOOPBACK_HOSTS:   # the safety rule: plain HTTP on this computer only
        return None                                            # any other computer is refused
    if port is None or not 1 <= port <= 65535:                 # a node is identified by its port, so one must be given
        return None                                            # unusable without a valid port
    return f"127.0.0.1:{port}"                                 # one agreed spelling, so the same node is never listed twice


def short_name(address: str) -> str:
    """Shorten "127.0.0.1:5001" to ":5001" for compact log lines."""
    return ":" + address.rsplit(":", 1)[-1]   # keep only the text after the last colon, which is the port


def call_node(method: str, address: str, path: str, payload=None) -> tuple:
    """Send one HTTP request to a node. Return (status code, decoded JSON), or (None, None) if it could not be reached."""
    url = f"http://{address}{path}"                             # build the full web address of the endpoint
    try:                                                        # the other node may be switched off or slow
        with requests.Session() as session:                     # a session is one short-lived connection manager
            session.trust_env = False                           # ignore any proxy settings: local nodes are reached directly
            response = session.request(method, url, json=payload, timeout=REQUEST_TIMEOUT)   # send the request and wait for the reply
        return response.status_code, response.json()            # hand back the status code and the decoded body
    except (requests.RequestException, ValueError):             # connection problems, timeouts and non-JSON replies land here
        return None, None                                       # tell the caller the node could not be used

# ===========================================================================
# THE PEER LIST
# ===========================================================================

class PeerBook:
    """The set of other nodes this node knows about."""

    def __init__(self, own_address: str):
        """Create an empty peer list for the node living at own_address."""
        self.own_address = own_address      # remembered so the node never lists itself as a peer
        self._peers = set()                 # a set holds each address at most once
        self._lock = threading.Lock()       # protects the set when several requests arrive together

    def add(self, address: str) -> str:
        """Try to add an already normalised address. Return "added", "known" or "full"."""
        with self._lock:                                              # only one request may change the list at a time
            if address == self.own_address or address in self._peers: # the node itself and existing peers are nothing new
                return "known"                                        # report that nothing changed
            if len(self._peers) >= MAX_PEERS:                         # the list has reached its limit
                return "full"                                         # report that there is no room
            self._peers.add(address)                                  # a new peer joins the list
            return "added"                                            # report the change

    def contains(self, address: str) -> bool:
        """Return True if the address is a known peer."""
        with self._lock:                      # read under the lock so the answer is consistent
            return address in self._peers     # True when the address is on the list

    def all(self) -> list:
        """Return every known peer in sorted order, so output is the same on every run."""
        with self._lock:                      # read under the lock so the list is not changing while it is copied
            return sorted(self._peers)        # a sorted copy of the set

# ===========================================================================
# ONE NODE: LEDGER, PEERS AND GOSSIP
# ===========================================================================

class Node:
    """One participant in the network. The Application Programming Interface (API) in api.py is a thin layer over the methods of this class."""

    def __init__(self, port: int):
        """Create a node that will listen on the given local port."""
        self.address = f"127.0.0.1:{port}"       # how other nodes reach this one
        self.default_miner = f"miner-{port}"     # the address rewarded when a mining request names nobody
        self.blockchain = Blockchain()           # this node's own copy of the ledger
        self.peers = PeerBook(self.address)      # this node's own list of peers
        self.lock = threading.Lock()             # protects the ledger when several requests arrive together
        self.events = []                         # a log of every gossip message sent or received

    # -----------------------------------------------------------------------
    # Logging
    # -----------------------------------------------------------------------
    def record(self, text: str):
        """Add one line to this node's gossip log."""
        entry = {"seq": next(EVENT_COUNTER), "node": self.address, "event": text}   # number the entry so logs can be merged in order
        self.events.append(entry)                                                   # store it
        del self.events[:-MAX_EVENTS]                                               # forget the oldest entries beyond the limit

    # -----------------------------------------------------------------------
    # Peer discovery: the counterpart of Bitcoin's version, getaddr and addr messages
    # -----------------------------------------------------------------------
    def register_peers(self, addresses: list, sender=None) -> dict:
        """Add peers, then pass on whatever was new. Return what happened to each address."""
        sender_address = normalise_address(sender)               # the node that sent this message, if it named itself
        candidates = list(addresses)                             # start with the addresses in the message
        if sender_address is not None:                           # a named sender introduces itself, like a version message
            candidates.append(sender_address)                    # so it is treated as one more address to learn
        added, already_known, rejected = [], [], []              # three lists describing the outcome
        for raw in candidates:                                   # consider each address in turn
            address = normalise_address(raw)                     # convert it to the agreed spelling
            if address is None:                                  # it was malformed or pointed at another computer
                rejected.append({"address": str(raw), "reason": "not an http address on this computer (127.0.0.1 or localhost) with a port"})   # record why
                continue                                         # move on to the next address
            if address in added or address in already_known:     # the same address listed twice is only handled once
                continue                                         # move on to the next address
            outcome = self.peers.add(address)                    # try to add it to the peer list
            if outcome == "added":                               # it was new
                added.append(address)                            # remember it for the relay step below
            elif outcome == "known":                             # it was already listed, or it is this node itself
                already_known.append(address)                    # nothing changes
            else:                                                # the only other outcome is a full list
                rejected.append({"address": address, "reason": f"peer list is full ({MAX_PEERS} peers)"})   # record why
        origin = short_name(sender_address) if sender_address else "a client"   # who sent the message, for the log
        if added:                                                # something new was learned
            learned = ", ".join(short_name(address) for address in added)       # a compact list for the log
            self.record(f"got addr from {origin}: learned {learned}")           # log the discovery
        elif sender_address is not None:                         # a peer told us only what we already knew
            self.record(f"got addr from {origin}: nothing new, so nothing relayed")   # this is what ends the gossip
        for peer in self.peers.all():                            # decide what, if anything, each peer should be told
            if peer == sender_address:                           # the node that just told us needs nothing back
                continue                                         # move on to the next peer
            news = [address for address in added if address != peer]   # the new addresses, leaving out the peer's own
            if peer in added:                                    # a brand new peer does not know this node yet
                news.insert(0, self.address)                     # so this node advertises its own address first
            if not news:                                         # nothing was new for this peer
                continue                                         # move on to the next peer
            names = ", ".join(short_name(address) for address in news)          # a compact list for the log
            self.record(f"sent addr ({names}) to {short_name(peer)}")           # log before sending, so the order is causal
            call_node("POST", peer, "/nodes/register", {"nodes": news, "from": self.address})   # the relay itself
        return {                                                 # the outcome, reported back to the caller
            "added": added,                                      # addresses that were new to this node
            "already_known": already_known,                      # addresses it already had
            "rejected": rejected,                                # addresses it refused, each with a reason
            "nodes": self.peers.all(),                           # the full peer list after any relaying
        }

    # -----------------------------------------------------------------------
    # Inventory gossip: the counterpart of Bitcoin's inv and getdata messages
    # -----------------------------------------------------------------------
    def announce(self, kind: str, item_hash: str, exclude=None) -> list:
        """Tell every peer "I hold this item" by sending its hash only. Return the peers that answered."""
        reached = []                                             # the peers that replied
        for peer in self.peers.all():                            # announce to every peer
            if peer == exclude:                                  # except the one the item came from
                continue                                         # it already holds the item
            self.record(f"sent inv ({kind} {item_hash[:8]}) to {short_name(peer)}")   # log before sending, so the order is causal
            status, _ = call_node("POST", peer, "/inv", {"type": kind, "hash": item_hash, "from": self.address})   # the announcement itself
            if status is None:                                   # the peer did not answer
                self.record(f"{short_name(peer)} could not be reached")   # log the failure and carry on
            else:                                                # the peer answered
                reached.append(peer)                             # count it as reached
        return reached                                           # the list of peers that were told

    def holds(self, kind: str, item_hash: str) -> bool:
        """Return True if this node already holds the transaction or block with this hash."""
        if kind == "block":                                                  # a block is looked up in the chain
            return self.blockchain.get_block(item_hash) is not None          # True when the chain holds that hash
        return self.blockchain.find_transaction(item_hash) is not None       # a transaction may be mined or pending

    def get_item(self, kind: str, item_hash: str):
        """Return the full transaction or block for a getdata request, or None if this node does not hold it."""
        with self.lock:                                                      # read the ledger while nothing is changing it
            if kind == "block":                                              # a block was asked for
                block = self.blockchain.get_block(item_hash)                 # look it up by hash
                item = block.to_dict() if block is not None else None        # convert it to its network form
            else:                                                            # a transaction was asked for
                found = self.blockchain.find_transaction(item_hash)          # look it up by identifier
                item = found["transaction"] if found is not None else None   # keep just the transaction
        if item is not None:                                                 # only log requests that were answered
            self.record(f"got getdata ({kind} {item_hash[:8]}), sent the {kind}")   # log the reply
        return item                                                          # the item, or None

    def receive_inventory(self, kind, item_hash, sender) -> tuple:
        """Handle an inv message: fetch the item if it is new, check it, store it and pass the news on.

        Return (HTTP status code, reply)."""
        sender_address = normalise_address(sender)                           # who is announcing
        if sender_address is None or not self.peers.contains(sender_address):   # announcements are only taken from known peers
            return 403, {"error": "inventory is only accepted from registered peers"}   # 403 means "forbidden"
        if kind not in ("tx", "block") or not isinstance(item_hash, str) or len(item_hash) != 64:   # the message must name a type and a 64 character hash
            return 400, {"error": "'type' must be \"tx\" or \"block\" and 'hash' must be 64 hexadecimal characters"}   # 400 means "bad request"
        label = f"{kind} {item_hash[:8]}"                                    # a compact name for the log
        with self.lock:                                                      # read the ledger while nothing is changing it
            already_held = self.holds(kind, item_hash)                       # is this item new to us?
        if already_held:                                                     # known items are ignored, which is what ends the gossip
            self.record(f"got inv ({label}) from {short_name(sender_address)}: already held, ignored")   # log it
            return 200, {"action": "ignored", "detail": "already held"}      # nothing more to do
        self.record(f"got inv ({label}) from {short_name(sender_address)}: new, sent getdata")   # log the decision to fetch
        status, data = call_node("GET", sender_address, f"/getdata/{kind}/{item_hash}")   # ask the announcer for the full item
        if status != 200:                                                    # the announcer could not supply it
            self.record(f"getdata ({label}) failed")                         # log the failure
            return 200, {"action": "failed", "detail": "the announcing node did not supply the item"}   # report it
        if kind == "tx":                                                     # the item is a payment
            return self._receive_payment(data, item_hash, sender_address)    # handle it as a payment
        return self._receive_block(data, item_hash, sender_address)          # otherwise handle it as a block

    def _receive_payment(self, data, item_hash: str, sender_address: str) -> tuple:
        """Check a payment fetched from a peer and add it to the pending pool."""
        if not isinstance(data, dict) or data.get("txid") != item_hash:      # the item must be the one that was announced
            return 200, {"action": "rejected", "detail": "the item does not match the announced hash"}   # refuse a swapped item
        with self.lock:                                                      # change the ledger safely
            outcome, message, _ = self.blockchain.add_payment(data)          # apply the same checks as for a client's payment
        if outcome != "accepted":                                            # the payment failed a check here
            self.record(f"tx {item_hash[:8]} refused: {message}")            # log why
            return 200, {"action": "rejected", "detail": message}            # report it
        self.record(f"tx {item_hash[:8]} added to the pending pool")         # log the success
        self.announce("tx", item_hash, exclude=sender_address)               # pass the news on to the other peers
        return 200, {"action": "fetched", "detail": "payment added to the pending pool"}   # report it

    def _receive_block(self, data, item_hash: str, sender_address: str) -> tuple:
        """Check a block fetched from a peer and add it to the chain, or fall back to comparing whole chains."""
        block = Block.from_dict(data)                                        # rebuild the block from the reply
        if block is None or block.hash != item_hash:                         # the item must be the one that was announced
            return 200, {"action": "rejected", "detail": "the item does not match the announced hash"}   # refuse a swapped item
        with self.lock:                                                      # change the ledger safely
            outcome, message = self.blockchain.accept_block(block)           # apply every block rule
        if outcome == "accepted":                                            # the block extended our chain
            self.record(f"block {block.index} ({item_hash[:8]}) appended to the chain")   # log the success
            self.announce("block", item_hash, exclude=sender_address)        # pass the news on to the other peers
            return 200, {"action": "fetched", "detail": "block appended to the chain"}   # report it
        if outcome == "detached":                                            # the block builds on something we do not hold
            self.record(f"block {block.index} ({item_hash[:8]}) does not attach, requested the whole chain")   # log the fallback
            replaced, message = self.sync_from(sender_address)               # compare whole chains with the announcer
            if replaced:                                                     # we switched to the announcer's chain
                self.announce("block", item_hash, exclude=sender_address)    # so tell the other peers about its newest block
                return 200, {"action": "synchronised", "detail": message}    # report it
        self.record(f"block {block.index} ({item_hash[:8]}) not adopted: {message}")   # every remaining case leaves our chain unchanged
        return 200, {"action": "rejected", "detail": message}                # report it

    # -----------------------------------------------------------------------
    # Chain synchronisation: the counterpart of Bitcoin's initial block download
    # -----------------------------------------------------------------------
    def sync_from(self, peer: str) -> tuple:
        """Download a peer's whole chain and adopt it if it is valid and longer. Return (replaced, message)."""
        status, data = call_node("GET", peer, "/chain")                          # ask for the peer's full chain
        if status != 200 or not isinstance(data, dict) or not isinstance(data.get("chain"), list):   # the reply must contain a list of blocks
            return False, "the peer did not supply a chain"                      # nothing to compare
        blocks = [Block.from_dict(item) for item in data["chain"]]               # rebuild every block (unreadable ones become None)
        with self.lock:                                                          # change the ledger safely
            replaced, message = self.blockchain.replace_chain(blocks)            # the longest valid chain rule
        if replaced:                                                             # only a switch is worth logging
            self.record(f"adopted the chain of {short_name(peer)}, now {len(blocks)} blocks")   # log it
        return replaced, message                                                 # report the outcome

    def resolve(self) -> dict:
        """Compare chains with every peer and keep the longest valid one."""
        checked = []                                                 # one line of results per peer
        for peer in self.peers.all():                                # visit every peer in turn
            replaced, message = self.sync_from(peer)                 # adopt its chain if it is longer and valid
            checked.append({"peer": peer, "adopted": replaced, "detail": message})   # record what happened
        with self.lock:                                              # read the ledger while nothing is changing it
            length = len(self.blockchain.chain)                      # the chain length after all comparisons
        replaced_any = any(item["adopted"] for item in checked)      # did any peer have a better chain?
        return {                                                     # the summary returned to the caller
            "replaced": replaced_any,                                # True if this node switched chains
            "length": length,                                        # how many blocks it holds now
            "checked": checked,                                      # the detail for each peer
        }

    # -----------------------------------------------------------------------
    # Actions requested by a client
    # -----------------------------------------------------------------------
    def submit_payment(self, data) -> tuple:
        """Accept a payment from a client and announce it to the peers. Return (outcome, message, payment, peers told)."""
        with self.lock:                                                  # change the ledger safely
            outcome, message, payment = self.blockchain.add_payment(data)   # validate the payment and add it to the pool
        announced_to = []                                                # the peers told about it, empty unless it was accepted
        if outcome == "accepted":                                        # only accepted payments are announced
            self.record(f"tx {payment['txid'][:8]} submitted by a client, added to the pending pool")   # log it
            announced_to = self.announce("tx", payment["txid"])          # tell the peers
        return outcome, message, payment, announced_to                   # report everything to the caller

    def mine(self, miner_address=None) -> tuple:
        """Mine the pending payments into a block and announce it. Return (block, attempts, seconds, peers told)."""
        miner = miner_address if miner_address is not None else self.default_miner   # use the node's own miner address unless one was given
        if not is_address(miner):                                        # the reward must go to a usable address
            return None, 0, 0.0, []                                      # None tells the caller the address was refused
        with self.lock:                                                  # hold the ledger still while the block is mined
            block, attempts, seconds = self.blockchain.mine_pending(miner)   # the proof-of-work search
        self.record(f"mined block {block.index} ({block.hash[:8]}) after {attempts:,} attempts")   # log it
        announced_to = self.announce("block", block.hash)                # tell the peers
        return block, attempts, seconds, announced_to                    # report everything to the caller

    def snapshot(self) -> dict:
        """Return everything the explorer needs in one consistent reading of the node."""
        with self.lock:                                                          # read the ledger while nothing is changing it
            chain = [block.to_dict() for block in self.blockchain.chain]         # every block in its network form
            pending = list(self.blockchain.pending)                              # a copy of the pending pool
        return {                                                                 # the snapshot
            "node": self.address,                                                # which node this describes
            "difficulty": DIFFICULTY,                                            # the mining rule every node shares
            "chain": chain,                                                      # its blocks
            "pending": pending,                                                  # its pending payments
            "nodes": self.peers.all(),                                           # its peers
        }

Overwriting network.py


## 4. The API: `api.py`

One Flask endpoint per action. The server binds to `127.0.0.1`, so it is reachable from this computer only.

In [4]:
%%writefile api.py
"""
api.py
Assignment 8: REST API and Networking Lab (the web interface of one node)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS PROGRAM DOES
    Wraps one Node (see network.py) in a REST (Representational State Transfer) API
    built with Flask. Each endpoint is a web address that reads or changes the node:

        Required by the brief
            GET  /chain                  the full chain
            POST /transactions/new       submit a payment
            POST /mine                   mine a block holding the pending payments
            GET  /nodes                  this node's peer list
            POST /nodes/register         add peers
        Supporting the payment flow
            GET  /                       a short status report and this list
            GET  /transactions/pending   payments waiting to be mined
            GET  /transactions/<txid>    one transaction and its confirmations
            GET  /balance/<address>      what an address holds and can spend
        Supporting the network (Part B)
            GET  /nodes/resolve          adopt the longest valid chain among the peers
            POST /inv                    "I hold this hash" from a peer
            GET  /getdata/<type>/<hash>  the full transaction or block behind a hash
            GET  /log                    this node's gossip log
        Explorer
            GET  /explorer               a web page listing recent blocks and transactions

    Every reply except /explorer is JSON (JavaScript Object Notation). The exact
    request and reply of every endpoint is documented in README.md.

SAFETY
    The server binds to 127.0.0.1, so it can only be reached from this computer. There
    is no login and payments are not signed, which is acceptable for a local
    demonstration only. Do not change the host to expose it on a network.

HOW TO RUN
    python api.py                 one node on port 5000
    python api.py --port 5001     a second node, in a second terminal
"""

import argparse                      # argparse reads options such as --port from the command line
import logging                       # logging lets us quieten the web server's request log when running in the background
import socket                        # socket lets us test whether a port is already taken
import threading                     # threading lets a node run in the background while other code carries on

from flask import Flask, Response, jsonify, request   # the pieces of Flask this file uses
from werkzeug.serving import make_server              # builds a web server object that can be stopped again

from blockchain import is_address                     # the address check from the chain core
from explorer import render_html                      # builds the explorer web page
from network import Node                              # one participant in the network

# ===========================================================================
# CONFIGURATION
# ===========================================================================

HOST = "127.0.0.1"       # the loopback address: reachable from this computer only
DEFAULT_PORT = 5000      # the port a node listens on unless told otherwise

ENDPOINTS = [                                                                   # the list returned by GET /
    "GET  /                       status of this node and this list",           # the index itself
    "GET  /chain                  the full chain",                              # required by the brief
    "POST /transactions/new       submit a payment",                            # required by the brief
    "GET  /transactions/pending   payments waiting to be mined",                # supports the payment flow
    "GET  /transactions/<txid>    one transaction and its confirmations",       # supports the payment flow
    "POST /mine                   mine a block holding the pending payments",   # required by the brief
    "GET  /balance/<address>      what an address holds and can spend",         # supports the payment flow
    "GET  /nodes                  this node's peer list",                       # required by the brief
    "POST /nodes/register         add peers",                                   # required by the brief
    "GET  /nodes/resolve          adopt the longest valid chain among the peers",   # supports the network
    "POST /inv                    announcement of a hash by a peer",            # supports the network
    "GET  /getdata/<type>/<hash>  the full transaction or block behind a hash", # supports the network
    "GET  /log                    this node's gossip log",                      # supports the network
    "GET  /explorer               web page of recent blocks and transactions",  # the explorer view
]

# ===========================================================================
# BUILDING THE FLASK APPLICATION
# ===========================================================================

def create_app(port: int = DEFAULT_PORT) -> Flask:
    """Create one node and the Flask application that exposes it."""
    node = Node(port)                              # the participant this application speaks for
    app = Flask(__name__)                          # the Flask application object
    app.config["NODE"] = node                      # keep a reference so other code can reach the node
    app.config["JSON_SORT_KEYS"] = False           # older Flask versions: keep reply fields in the order written below
    if hasattr(app, "json"):                       # newer Flask versions use a different setting for the same thing
        app.json.sort_keys = False                 # keep reply fields in the order written below

    # -----------------------------------------------------------------------
    # Status
    # -----------------------------------------------------------------------
    @app.route("/", methods=["GET"])               # GET / answers with a short status report
    def index():
        """Describe this node and list its endpoints."""
        view = node.snapshot()                               # one consistent reading of the node
        return jsonify({                                     # reply in JSON
            "node": node.address,                            # which node is answering
            "chain_length": len(view["chain"]),              # how many blocks it holds, including genesis
            "pending_count": len(view["pending"]),           # how many payments are waiting
            "peer_count": len(view["nodes"]),                # how many peers it knows
            "endpoints": ENDPOINTS,                          # what can be asked of it
        }), 200                                              # 200 means "OK"

    # -----------------------------------------------------------------------
    # The chain
    # -----------------------------------------------------------------------
    @app.route("/chain", methods=["GET"])          # GET /chain returns every block
    def get_chain():
        """Return the full chain."""
        view = node.snapshot()                               # one consistent reading of the node
        return jsonify({                                     # reply in JSON
            "length": len(view["chain"]),                    # how many blocks, including genesis
            "difficulty": view["difficulty"],                # the number of leading zeros every hash must show
            "chain": view["chain"],                          # the blocks, oldest first
        }), 200                                              # 200 means "OK"

    # -----------------------------------------------------------------------
    # Payments
    # -----------------------------------------------------------------------
    @app.route("/transactions/new", methods=["POST"])   # POST /transactions/new submits a payment
    def new_transaction():
        """Validate a payment, add it to the pending pool and announce it to the peers."""
        data = request.get_json(silent=True)                         # read the JSON body, or None if it is missing or malformed
        outcome, message, payment, announced_to = node.submit_payment(data)   # hand the payment to the node
        if outcome == "invalid":                                     # the fields were wrong
            return jsonify({"error": message}), 400                  # 400 means "bad request"
        if outcome == "duplicate":                                   # this exact payment was submitted before
            return jsonify({"error": message, "txid": payment["txid"]}), 409   # 409 means "conflict"
        if outcome == "insufficient_funds":                          # the sender cannot afford it
            return jsonify({"error": message}), 422                  # 422 means "understood, but cannot be processed"
        return jsonify({                                             # the payment was accepted
            "message": message,                                      # a sentence saying so
            "transaction": payment,                                  # the payment as stored, including its txid
            "pending_count": len(node.snapshot()["pending"]),        # how many payments are now waiting
            "announced_to": announced_to,                            # the peers that were told
        }), 201                                                      # 201 means "created"

    @app.route("/transactions/pending", methods=["GET"])   # GET /transactions/pending lists the pool
    def pending_transactions():
        """Return the payments waiting to be mined."""
        pending = node.snapshot()["pending"]                         # a copy of the pool
        return jsonify({"count": len(pending), "transactions": pending}), 200   # how many, then the payments themselves

    @app.route("/transactions/<txid>", methods=["GET"])    # GET /transactions/<txid> looks one transaction up
    def get_transaction(txid):
        """Return one transaction with its status and number of confirmations."""
        with node.lock:                                              # read the ledger while nothing is changing it
            found = node.blockchain.find_transaction(txid)           # search the chain, then the pool
        if found is None:                                            # this node has never seen that identifier
            return jsonify({"error": "no transaction with that txid is known to this node"}), 404   # 404 means "not found"
        return jsonify(found), 200                                   # the status report

    @app.route("/balance/<address>", methods=["GET"])      # GET /balance/<address> reports one balance
    def get_balance(address):
        """Return what an address holds, what is waiting to leave it, and what it can still spend."""
        if not is_address(address):                                  # refuse names that could never be an address
            return jsonify({"error": "an address is 1 to 64 letters, digits, hyphens or underscores"}), 400   # 400 means "bad request"
        with node.lock:                                              # read the ledger while nothing is changing it
            report = node.blockchain.balance_of(address)             # confirmed, pending out and available
        return jsonify(report), 200                                  # the balance report

    # -----------------------------------------------------------------------
    # Mining
    # -----------------------------------------------------------------------
    @app.route("/mine", methods=["POST"])          # POST /mine mines one block
    def mine():
        """Mine a block holding every pending payment, then announce it to the peers."""
        data = request.get_json(silent=True) or {}                   # the body is optional, so default to no fields
        miner = data.get("miner") if isinstance(data, dict) else None   # the address to reward, if one was named
        block, attempts, seconds, announced_to = node.mine(miner)    # the proof-of-work search happens here
        if block is None:                                            # the miner address was refused
            return jsonify({"error": "'miner' must be 1 to 64 letters, digits, hyphens or underscores"}), 400   # 400 means "bad request"
        return jsonify({                                             # the block was mined
            "message": f"block {block.index} mined",                 # a sentence saying so
            "block": block.to_dict(),                                # the new block
            "attempts": attempts,                                    # how many hashes the search needed
            "seconds": round(seconds, 3),                            # how long it took on this machine
            "announced_to": announced_to,                            # the peers that were told
        }), 201                                                      # 201 means "created"

    # -----------------------------------------------------------------------
    # Peers
    # -----------------------------------------------------------------------
    @app.route("/nodes", methods=["GET"])          # GET /nodes returns the peer list
    def get_nodes():
        """Return this node's address and its peers."""
        peers = node.peers.all()                                     # the sorted peer list
        return jsonify({"node": node.address, "count": len(peers), "nodes": peers}), 200   # who is answering, how many peers, and which

    @app.route("/nodes/register", methods=["POST"])   # POST /nodes/register adds peers
    def register_nodes():
        """Add one or more peers and relay the new ones to the rest of the network."""
        data = request.get_json(silent=True)                         # read the JSON body, or None if it is missing or malformed
        addresses = data.get("nodes") if isinstance(data, dict) else None   # the list of addresses to add
        if not isinstance(addresses, list) or not addresses:         # a non-empty list is required
            return jsonify({"error": "send a JSON object such as {\"nodes\": [\"http://127.0.0.1:5001\"]}"}), 400   # 400 means "bad request"
        result = node.register_peers(addresses, data.get("from"))    # add them and relay what was new
        if result["added"]:                                          # at least one peer was new
            result = {"message": f"{len(result['added'])} peer(s) added", **result}   # put a summary sentence first
            return jsonify(result), 201                              # 201 means "created"
        if result["rejected"]:                                       # nothing was added and something was refused
            result = {"error": "no peer was added", **result}        # put the error first
            return jsonify(result), 400                              # 400 means "bad request"
        result = {"message": "no new peers, all were already known", **result}   # nothing was new and nothing was wrong
        return jsonify(result), 200                                  # 200 means "OK"

    @app.route("/nodes/resolve", methods=["GET"])  # GET /nodes/resolve compares chains with the peers
    def resolve():
        """Adopt the longest valid chain held by any peer."""
        result = node.resolve()                                      # download and compare each peer's chain
        message = "chain replaced by a longer valid chain" if result["replaced"] else "this node's chain was kept"   # a summary sentence
        return jsonify({"message": message, **result}), 200          # 200 means "OK"

    # -----------------------------------------------------------------------
    # Gossip between nodes
    # -----------------------------------------------------------------------
    @app.route("/inv", methods=["POST"])           # POST /inv receives an announcement from a peer
    def inventory():
        """Receive "I hold this hash" from a peer, and fetch the item if it is new."""
        data = request.get_json(silent=True)                         # read the JSON body, or None if it is missing or malformed
        if not isinstance(data, dict):                               # the body must be a JSON object
            return jsonify({"error": "send a JSON object with 'type', 'hash' and 'from'"}), 400   # 400 means "bad request"
        status, reply = node.receive_inventory(data.get("type"), data.get("hash"), data.get("from"))   # let the node decide what to do
        return jsonify(reply), status                                # pass the node's answer back

    @app.route("/getdata/<kind>/<item_hash>", methods=["GET"])   # GET /getdata/<type>/<hash> supplies a full item
    def getdata(kind, item_hash):
        """Return the full transaction or block behind a hash."""
        if kind not in ("tx", "block"):                              # only two kinds of item exist
            return jsonify({"error": "the type must be \"tx\" or \"block\""}), 400   # 400 means "bad request"
        item = node.get_item(kind, item_hash)                        # look the item up
        if item is None:                                             # this node does not hold it
            return jsonify({"error": f"this node holds no {kind} with that hash"}), 404   # 404 means "not found"
        return jsonify(item), 200                                    # the transaction or block itself

    @app.route("/log", methods=["GET"])            # GET /log returns the gossip log
    def get_log():
        """Return every gossip message this node has sent or received."""
        events = list(node.events)                                   # a copy of the log
        return jsonify({"node": node.address, "count": len(events), "events": events}), 200   # who is answering, how many entries, and which

    # -----------------------------------------------------------------------
    # Explorer page
    # -----------------------------------------------------------------------
    @app.route("/explorer", methods=["GET"])       # GET /explorer returns a web page instead of JSON
    def explorer_page():
        """Return the block explorer as a HyperText Markup Language (HTML) page."""
        page = render_html(node.snapshot())                          # build the page from a reading of the node
        return Response(page, mimetype="text/html")                  # label the reply as a web page

    # -----------------------------------------------------------------------
    # Errors
    # -----------------------------------------------------------------------
    @app.errorhandler(404)                         # runs when no endpoint matches the web address
    def not_found(_error):
        """Answer unknown addresses in JSON, like every other reply."""
        return jsonify({"error": "no such endpoint; GET / lists the endpoints"}), 404   # 404 means "not found"

    @app.errorhandler(405)                         # runs when the address exists but the method is wrong
    def wrong_method(_error):
        """Answer a wrong method (for example GET /mine) in JSON, like every other reply."""
        return jsonify({"error": "this endpoint does not accept that method; GET / lists the endpoints"}), 405   # 405 means "method not allowed"

    return app                                     # hand back the finished application

# ===========================================================================
# RUNNING NODES IN THE BACKGROUND (USED BY THE NOTEBOOK AND THE DEMONSTRATION SCRIPT)
# ===========================================================================

RUNNING = globals().get("RUNNING", {})   # background nodes by port; kept if this file is reloaded, so none are lost track of


def port_in_use(port: int) -> bool:
    """Return True if something on this computer is already listening on the port."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:   # open a temporary network socket
        probe.settimeout(0.5)                                          # give up after half a second
        return probe.connect_ex((HOST, port)) == 0                     # 0 means the connection succeeded, so the port is taken


class BackgroundNode:
    """One node running inside this program on a background thread."""

    def __init__(self, port: int):
        """Create the node and start serving it straight away."""
        self.port = port                                             # the port this node listens on
        self.url = f"http://{HOST}:{port}"                           # the base web address of the node
        self.app = create_app(port)                                  # the Flask application
        self.node = self.app.config["NODE"]                          # the node behind it
        self.server = make_server(HOST, port, self.app, threaded=True)   # a web server that handles each request on its own thread
        self.thread = threading.Thread(target=self.server.serve_forever, daemon=True)   # daemon=True lets the program exit without waiting
        self.thread.start()                                          # begin answering requests

    def stop(self):
        """Stop serving and release the port."""
        self.server.shutdown()             # ask the serving loop to finish
        self.server.server_close()         # close the listening socket so the port is free again
        self.thread.join()                 # wait for the background thread to end


def start_background_node(port: int) -> BackgroundNode:
    """Start a fresh node on the port, replacing any background node already running there."""
    if port in RUNNING:                                              # a node from an earlier run is still on this port
        RUNNING.pop(port).stop()                                     # stop it so the new node starts from a clean ledger
    if port_in_use(port):                                            # some other program holds the port
        raise RuntimeError(f"port {port} is already in use; close the program using it or choose another port")   # explain how to fix it
    logging.getLogger("werkzeug").setLevel(logging.ERROR)            # keep the web server's request log out of the output
    RUNNING[port] = BackgroundNode(port)                             # start the node and remember it
    return RUNNING[port]                                             # hand it back to the caller


def stop_background_nodes():
    """Stop every background node this program started."""
    for port in list(RUNNING):             # list() copies the ports, because the dictionary shrinks as we go
        RUNNING.pop(port).stop()           # stop each node and forget it

# ===========================================================================
# RUNNING ONE NODE FROM THE COMMAND LINE
# ===========================================================================

def main():
    """Read the port from the command line and serve one node until Ctrl+C is pressed."""
    parser = argparse.ArgumentParser(description="Run one blockchain node on this computer.")   # describes the program for --help
    parser.add_argument("--port", type=int, default=DEFAULT_PORT, help="port to listen on (default 5000)")   # the only option
    arguments = parser.parse_args()                                  # read what was typed
    if port_in_use(arguments.port):                                  # refuse to start two nodes on one port
        raise SystemExit(f"Port {arguments.port} is already in use. Choose another with --port.")   # stop with a clear message
    app = create_app(arguments.port)                                 # build the node and its application
    print(f"Node listening on http://{HOST}:{arguments.port}  (explorer at http://{HOST}:{arguments.port}/explorer)")   # where to find it
    app.run(host=HOST, port=arguments.port, threaded=True)           # serve on the loopback address only


if __name__ == "__main__":      # True only when this file is run directly, not when it is imported
    main()                      # start the node

Overwriting api.py


## 5. The explorer: `explorer.py`

One reading of a node, shown either as text or as a web page.

In [5]:
%%writefile explorer.py
"""
explorer.py
Assignment 8: REST API and Networking Lab (the block explorer)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS PROGRAM DOES
    Shows what a node holds, in the way a public block explorer does: a summary of the
    node, its recent blocks, its recent transactions, its pending payments and the
    balance of every address. The same information is produced in two forms:

        render_text   a plain text report for the terminal
        render_html   a HyperText Markup Language (HTML) page, served by the node itself
                      at /explorer and also written to a file with --html

    Run from the command line, this program is a client of the API (Application
    Programming Interface) in api.py: it only reads GET /chain,
    GET /transactions/pending and GET /nodes, so it sees exactly what any other
    client of the node would see.

HOW TO RUN (a node must already be running, see api.py)
    python explorer.py                                 the node on port 5000
    python explorer.py --node http://127.0.0.1:5001    a different node
    python explorer.py --last 10                       show up to 10 rows per table
    python explorer.py --html explorer.html            also save the page to a file
"""

import argparse                              # argparse reads options such as --node from the command line
import html                                  # html.escape makes text safe to place inside a web page
from datetime import datetime, timezone      # used to turn a Unix timestamp into a readable date and time

from blockchain import apply_transaction     # the same balance arithmetic the nodes use
from network import call_node, normalise_address   # the request helper and the local-address check

# ===========================================================================
# CONFIGURATION
# ===========================================================================

DEFAULT_NODE = "http://127.0.0.1:5000"   # the node the explorer reads unless told otherwise
DEFAULT_ROWS = 5                         # how many recent blocks and transactions to list
HASH_CHARACTERS = 16                     # how much of each 64 character hash to display

# ===========================================================================
# READING A NODE THROUGH ITS API
# ===========================================================================

def fetch_snapshot(node_url: str = DEFAULT_NODE):
    """Read a node over HTTP (HyperText Transfer Protocol). Return a snapshot, or None if it cannot be reached."""
    address = normalise_address(node_url)                                  # convert the address to the agreed spelling
    if address is None:                                                    # the explorer only reads nodes on this computer
        return None                                                        # an unusable address gives no snapshot
    chain_status, chain_reply = call_node("GET", address, "/chain")        # the blocks
    pending_status, pending_reply = call_node("GET", address, "/transactions/pending")   # the pending payments
    nodes_status, nodes_reply = call_node("GET", address, "/nodes")        # the peers
    if (chain_status, pending_status, nodes_status) != (200, 200, 200):    # all three requests must succeed
        return None                                                        # otherwise there is nothing reliable to show
    return {                                                               # the same shape that Node.snapshot() produces
        "node": address,                                                   # which node was read
        "difficulty": chain_reply["difficulty"],                           # its mining rule
        "chain": chain_reply["chain"],                                     # its blocks
        "pending": pending_reply["transactions"],                          # its pending payments
        "nodes": nodes_reply["nodes"],                                     # its peers
    }

# ===========================================================================
# TURNING A SNAPSHOT INTO ROWS
# ===========================================================================

def shorten(hash_hex: str) -> str:
    """Show the start of a hash followed by three dots, so tables stay narrow."""
    return hash_hex[:HASH_CHARACTERS] + "..."     # the leading characters are enough to recognise a hash


def format_time(timestamp: int) -> str:
    """Turn a Unix timestamp into a date and time in UTC (Coordinated Universal Time)."""
    moment = datetime.fromtimestamp(timestamp, tz=timezone.utc)   # interpret the number of seconds as a UTC moment
    return moment.strftime("%Y-%m-%d %H:%M")                      # for example "2026-10-01 00:10"


def transaction_row(place: str, transaction: dict) -> list:
    """Describe one transaction as a list of table cells."""
    return [                                                      # one cell per column
        place,                                                    # the block number, or the word "pending"
        transaction["type"],                                      # "coinbase" or "payment"
        transaction["sender"] or "(new coins)",                   # a coinbase has no sender, so say where the value came from
        transaction["receiver"],                                  # who was paid
        f"{transaction['amount']:,}",                             # the amount, with thousands separators
        str(transaction["fee"]),                                  # the fee paid to the miner
        transaction["reference"],                                 # the payer's reference
        shorten(transaction["txid"]),                             # the start of the identifier
    ]


def build_view(snapshot: dict, last: int = DEFAULT_ROWS) -> tuple:
    """Reduce a snapshot to the summary facts and the tables that both views display."""
    chain, pending, peers = snapshot["chain"], snapshot["pending"], snapshot["nodes"]   # the three things a node holds
    facts = [                                                                  # the summary shown at the top
        ("Node", snapshot["node"]),                                            # which node this is
        ("Chain height", f"{len(chain) - 1} ({len(chain)} blocks including genesis)"),   # height counts from zero
        ("Difficulty", f"{snapshot['difficulty']} leading zeros"),             # the mining rule
        ("Pending payments", str(len(pending))),                               # how many payments are waiting
        ("Peers", ", ".join(peers) if peers else "none"),                      # the peers, or the word "none"
    ]
    block_rows = [                                                             # one row per recent block
        [                                                                      # one cell per column
            str(block["index"]),                                               # position in the chain
            format_time(block["timestamp"]),                                   # time on the logical clock
            str(len(block["transactions"])),                                   # how many transactions it holds
            f"{block['nonce']:,}",                                             # the nonce the miner found
            shorten(block["hash"]),                                            # the start of its hash
            shorten(block["previous_hash"]),                                   # the start of its parent's hash
        ]
        for block in reversed(chain[-last:])                                   # the last few blocks, newest first
    ]
    mined = [(block["index"], transaction)                                     # pair every mined transaction with its block number
             for block in chain                                                # walking the chain from genesis
             for transaction in block["transactions"]]                         # and every transaction inside each block
    transaction_rows = [transaction_row(str(index), transaction)               # one row per recent transaction
                        for index, transaction in reversed(mined[-last:])]     # the last few, newest first
    pending_rows = [transaction_row("pending", payment) for payment in pending]   # one row per waiting payment
    balances = {}                                                              # work out every balance from the chain
    for _, transaction in mined:                                               # replay every mined transaction
        apply_transaction(balances, transaction)                               # move its value
    balance_rows = [[address, f"{value:,}"] for address, value in sorted(balances.items())]   # one row per address, in name order
    transaction_headers = ["Block", "Type", "From", "To", "Amount", "Fee", "Reference", "Txid"]   # shared by two tables
    sections = [                                                               # each table: a title, column headings and rows
        (f"RECENT BLOCKS (newest first, {len(block_rows)} of {len(chain)})",   # the title says how many are shown
         ["Index", "Time (UTC)", "Txs", "Nonce", "Hash", "Previous hash"], block_rows),   # headings, then the rows
        (f"RECENT TRANSACTIONS (newest first, {len(transaction_rows)} of {len(mined)})",   # the title says how many are shown
         transaction_headers, transaction_rows),                               # headings, then the rows
        (f"PENDING PAYMENTS ({len(pending_rows)})", transaction_headers, pending_rows),   # the waiting payments
        ("CONFIRMED BALANCES", ["Address", "Balance"], balance_rows),          # what each address holds
    ]
    return facts, sections                                                     # everything the two views need

# ===========================================================================
# THE TEXT VIEW
# ===========================================================================

def render_text(snapshot: dict, last: int = DEFAULT_ROWS) -> str:
    """Return the explorer as plain text for the terminal."""
    facts, sections = build_view(snapshot, last)                               # the facts and tables to display
    lines = ["=" * 100, f"BLOCK EXPLORER  node {snapshot['node']}", "=" * 100]   # a heading between two rules
    for name, value in facts:                                                  # the summary, one fact per line
        lines.append(f"{name:<17}: {value}")                                   # "<17" pads the name so the colons line up
    for title, headers, rows in sections:                                      # then each table in turn
        lines.append("")                                                       # a blank line before the table
        lines.append(title)                                                    # the table's title
        if not rows:                                                           # an empty table is shown as one word
            lines.append("  (none)")                                           # so the reader knows nothing is missing
            continue                                                           # move on to the next table
        widths = [max(len(row[column]) for row in [headers] + rows)            # each column is as wide as its longest cell
                  for column in range(len(headers))]                           # worked out column by column
        for row in [headers] + rows:                                           # print the headings, then every row
            cells = [cell.ljust(width) for cell, width in zip(row, widths)]    # pad each cell to its column width
            lines.append("  " + "  ".join(cells).rstrip())                     # join the cells with two spaces between columns
    return "\n".join(lines)                                                    # one string with a line break between lines

# ===========================================================================
# THE WEB PAGE VIEW
# ===========================================================================

# The styling rules for the page, written in CSS (Cascading Style Sheets). The page is black on white only,
# and every rule starts with .bx so the styling cannot leak into a surrounding page such as a notebook.
# "!important" keeps the table looking the same when a notebook applies its own table styling.
PAGE_STYLE = """
.bx { font-family: Calibri, Arial, sans-serif; color: #000; background: #fff; max-width: 1100px; margin: 0 auto; padding: 16px; }
.bx h1 { font-size: 22px; margin: 0 0 12px 0; }
.bx h2 { font-size: 15px; margin: 22px 0 6px 0; }
.bx table { border-collapse: collapse; width: 100%; font-size: 13px; table-layout: auto !important; margin: 0 !important; }
.bx th, .bx td { border: 1px solid #000 !important; padding: 4px 8px; text-align: left !important; color: #000; background: #fff; }
.bx th { font-weight: bold; }
.bx td.mono { font-family: Consolas, "Courier New", monospace; white-space: nowrap; }
.bx p { font-size: 13px; margin: 4px 0; }
"""


def html_table(headers: list, rows: list) -> str:
    """Return one HTML table. Every cell is escaped, so text from a transaction can never act as page code."""
    if not rows:                                                               # an empty table is shown as one word
        return "<p>(none)</p>"                                                 # so the reader knows nothing is missing
    head = "".join(f"<th>{html.escape(cell)}</th>" for cell in headers)        # the heading cells
    body = ""                                                                  # the rows are added to this text one by one
    for row in rows:                                                           # each row of the table
        cells = ""                                                             # the cells of this row
        for header, cell in zip(headers, row):                                 # each cell together with its column heading
            style = ' class="mono"' if header in ("Hash", "Previous hash", "Txid") else ""   # hashes use a fixed-width font
            cells += f"<td{style}>{html.escape(cell)}</td>"                    # add the escaped cell
        body += f"<tr>{cells}</tr>"                                            # close the row
    return f"<table><tr>{head}</tr>{body}</table>"                             # the finished table


def render_html(snapshot: dict, last: int = DEFAULT_ROWS) -> str:
    """Return the explorer as a self-contained web page."""
    facts, sections = build_view(snapshot, last)                               # the same facts and tables as the text view
    fact_rows = [[name, value] for name, value in facts]                       # the summary becomes a two column table
    parts = [f"<h1>Block explorer, node {html.escape(snapshot['node'])}</h1>"] # the page heading
    parts.append(html_table(["Item", "Value"], fact_rows))                     # the summary table
    for title, headers, rows in sections:                                      # then each table in turn
        parts.append(f"<h2>{html.escape(title)}</h2>")                         # the table's title
        parts.append(html_table(headers, rows))                                # the table itself
    content = "\n".join(parts)                                                 # all the pieces, one per line
    return (                                                                   # wrap the pieces in a complete page
        "<!DOCTYPE html>\n"                                                    # tells the browser this is modern HTML
        '<html lang="en"><head><meta charset="utf-8">\n'                       # the language and the character set
        f"<title>Block explorer {html.escape(snapshot['node'])}</title>\n"     # the text shown on the browser tab
        f"<style>{PAGE_STYLE}</style></head>\n"                                # the styling rules above
        f'<body><div class="bx">\n{content}\n</div></body></html>\n'           # the visible content
    )

# ===========================================================================
# RUNNING THE EXPLORER FROM THE COMMAND LINE
# ===========================================================================

def main():
    """Read a running node and print the explorer."""
    parser = argparse.ArgumentParser(description="Print a block explorer view of a running node.")   # describes the program for --help
    parser.add_argument("--node", default=DEFAULT_NODE, help="address of the node to read (default http://127.0.0.1:5000)")   # which node
    parser.add_argument("--last", type=int, default=DEFAULT_ROWS, help="rows per table (default 5)")   # how much to show
    parser.add_argument("--html", metavar="FILE", help="also save the view as a web page in FILE")     # optional page output
    arguments = parser.parse_args()                                            # read what was typed
    snapshot = fetch_snapshot(arguments.node)                                  # read the node through its API
    if snapshot is None:                                                       # the node did not answer
        raise SystemExit(f"No node answered at {arguments.node}. Start one with: python api.py")   # explain how to fix it
    rows = max(1, arguments.last)                                              # always show at least one row
    print(render_text(snapshot, rows))                                         # the text view
    if arguments.html:                                                         # a page was asked for as well
        with open(arguments.html, "w", encoding="utf-8") as page_file:         # open the file for writing
            page_file.write(render_html(snapshot, rows))                       # write the page
        print(f"\nWeb page written to {arguments.html}")                       # say where it went


if __name__ == "__main__":      # True only when this file is run directly, not when it is imported
    main()                      # print the explorer

Overwriting explorer.py


## 6. Three local nodes

Nodes A, B and C start on ports 5000, 5001 and 5002, each holding only the genesis block. `show` sends one request and prints the exchange.

In [6]:
for name in ("blockchain", "network", "explorer", "api"):               # the four files saved above, in the order they depend on each other
    importlib.reload(importlib.import_module(name))                     # load each file, and reload it in case it was edited and saved again

from api import start_background_node, stop_background_nodes            # start and stop nodes inside this notebook
from explorer import fetch_snapshot, render_html, render_text           # the explorer's reader and its two views
from network import call_node                                           # sends one request to a node

NODE_A, NODE_B, NODE_C = "127.0.0.1:5000", "127.0.0.1:5001", "127.0.0.1:5002"   # the addresses of the three nodes
NODES = [NODE_A, NODE_B, NODE_C]                                        # the same three, as a list to loop over

for address in NODES:                                                   # start each node in turn
    port = int(address.split(":")[1])                                   # the port is the number after the colon
    start_background_node(port)                                         # a fresh node, replacing any node left from an earlier run
    print("Node started at http://" + address)                          # confirm where it is listening


def show(method, address, path, body=None):
    """Send one request to a node, print the exchange, and return the reply."""
    status, reply = call_node(method, address, path, body)              # the request itself
    print(f"{method} http://{address}{path}")                           # what was asked
    if body is not None:                                                # only POST requests carry a body
        print("body:", json.dumps(body))                                # the JSON (JavaScript Object Notation) that was sent
    print("HTTP", status)                                               # the status code of the reply
    print(json.dumps(reply, indent=2))                                  # the reply, indented for reading
    print()                                                             # a blank line after each exchange
    return reply                                                        # hand the reply back so later cells can use it

RuntimeError: port 5000 is already in use; close the program using it or choose another port

## 7. A full payment on one node

### 7.1 The chain at the start

`GET /chain` returns the full chain. So far it holds only the genesis block.

In [ ]:
reply = show("GET", NODE_A, "/chain")   # read the full chain of node A

### 7.2 Alice is funded

No balance exists until a block is mined. Alice mines block 1 with `POST /mine` and receives the reward of 50.

In [ ]:
reply = show("POST", NODE_A, "/mine", {"miner": "alice"})   # mine one block, paying the reward to Alice
reply = show("GET", NODE_A, "/balance/alice")               # Alice's balance after the block

### 7.3 Alice pays Bob

`POST /transactions/new` submits the payment. The reply is 201 and carries the `txid`.

In [ ]:
payment = {"sender": "alice", "receiver": "bob", "amount": 20, "reference": "remittance-001"}   # the payment instruction
reply = show("POST", NODE_A, "/transactions/new", payment)                                       # submit it to node A
txid = reply["transaction"]["txid"]                                                              # keep the identifier for the next steps

### 7.4 The payment is pending

It sits in the pending pool, and 21 of Alice's 50 is reserved: the amount of 20 plus the fee of 1.

In [ ]:
reply = show("GET", NODE_A, "/transactions/pending")   # the payments waiting to be mined
reply = show("GET", NODE_A, "/balance/alice")          # confirmed, reserved and available amounts

### 7.5 A block is mined

`POST /mine` places the pending payment in block 2, after the miner's coinbase.

In [ ]:
reply = show("POST", NODE_A, "/mine", {"miner": "miner-a"})   # mine block 2, paying the reward to a separate miner

### 7.6 The payment is confirmed

One confirmation. Alice holds 29, Bob 20, and the miner 51: the subsidy of 50 plus the fee.

In [ ]:
reply = show("GET", NODE_A, f"/transactions/{txid}")          # look the payment up by its identifier
for name in ("alice", "bob", "miner-a"):                      # the three addresses involved
    status, balance = call_node("GET", NODE_A, f"/balance/{name}")   # read one balance
    print(f"{name:<8} confirmed balance {balance['confirmed']}")     # print it on one line

### 7.7 Three refusals

The same payment again (409), a payment Alice cannot afford (422), and an amount that is not a whole number (400).

In [ ]:
reply = show("POST", NODE_A, "/transactions/new", payment)                                                    # identical payment, so identical txid
reply = show("POST", NODE_A, "/transactions/new", {"sender": "alice", "receiver": "bob", "amount": 500})      # more than Alice has available
reply = show("POST", NODE_A, "/transactions/new", {"sender": "alice", "receiver": "bob", "amount": 2.5})      # decimals are not accepted

## 8. The explorer

### 8.1 Text view

The same output as `python explorer.py`.

In [ ]:
print(render_text(fetch_snapshot(NODE_A)))   # read node A through its API and print the text view

### 8.2 Web page view

The page the node serves at http://127.0.0.1:5000/explorer, shown here inline.

In [ ]:
display(HTML(render_html(fetch_snapshot(NODE_A))))   # read node A through its API and show the page

## 9. Three nodes

### 9.1 Peer discovery

A is told about B. C is then told about A only, and learns of B from the addresses the nodes pass on.

In [ ]:
last_seen = call_node("GET", NODE_A, "/log")[1]["events"][-1]["seq"]   # the newest log entry so far, so later cells show only what is new


def show_log(title):
    """Print the gossip messages of all three nodes that are new since the last call, in the order they happened."""
    global last_seen                                                    # this function moves the marker forward
    events = []                                                         # the entries of every node, to be merged
    for address in NODES:                                               # read each node's log
        events.extend(call_node("GET", address, "/log")[1]["events"])   # GET /log, keeping the entries
    events = sorted((entry for entry in events if entry["seq"] > last_seen), key=lambda entry: entry["seq"])   # the new entries, in order
    print(title)                                                        # a title for this slice of the log
    print("  seq  node   event")                                        # column headings
    for entry in events:                                                # one line per message
        print(f"  {entry['seq']:>3}  :{entry['node'].rsplit(':', 1)[-1]}  {entry['event']}")   # number, port and description
    if events:                                                          # move the marker only if something was shown
        last_seen = events[-1]["seq"]                                   # the next call starts after this entry


reply = show("POST", NODE_A, "/nodes/register", {"nodes": ["http://" + NODE_B]})   # node A is told about node B
reply = show("POST", NODE_C, "/nodes/register", {"nodes": ["http://" + NODE_A]})   # node C is told about node A only
for address in NODES:                                                               # every node's peer list
    print(address, "knows", call_node("GET", address, "/nodes")[1]["nodes"])       # GET /nodes
print()                                                                             # a blank line before the log
show_log("Address messages:")                                                       # how the addresses travelled

### 9.2 Catching up

B and C hold only the genesis block. `GET /nodes/resolve` adopts the longest valid chain among their peers.

In [ ]:
reply = show("GET", NODE_B, "/nodes/resolve")   # node B compares chains with its peers
reply = show("GET", NODE_C, "/nodes/resolve")   # node C does the same
show_log("Synchronisation messages:")           # what each node adopted

### 9.3 A payment spreads by `inv` and `getdata`

Bob pays Carol through node C. A and B are never contacted by the client, yet both end up holding the payment.

In [ ]:
second_payment = {"sender": "bob", "receiver": "carol", "amount": 5, "reference": "school-fees"}   # the second payment
reply = show("POST", NODE_C, "/transactions/new", second_payment)                                   # submitted to node C only
for address in NODES:                                                                               # every node's pending pool
    pool = call_node("GET", address, "/transactions/pending")[1]                                    # GET /transactions/pending
    print(address, "pending:", [item["txid"][:8] for item in pool["transactions"]])                 # the start of each txid
print()                                                                                             # a blank line before the log
show_log("Transaction gossip:")                                                                     # how the payment travelled

### 9.4 A block spreads the same way

Node B mines block 3. A and C fetch it, check it and append it.

In [ ]:
reply = show("POST", NODE_B, "/mine", {"miner": "miner-b"})   # mined at node B
show_log("Block gossip:")                                     # how the block travelled

### 9.5 All three nodes agree

In [ ]:
for address in NODES:                                                       # compare the three nodes
    chain = call_node("GET", address, "/chain")[1]                          # each node's chain
    pool = call_node("GET", address, "/transactions/pending")[1]            # and its pending pool
    print(address, "length", chain["length"], "newest block", chain["chain"][-1]["hash"], "pending", pool["count"])   # one line per node
print()                                                                     # a blank line before the explorer
print(render_text(fetch_snapshot(NODE_C)))                                  # the explorer's view of node C

## 10. Checks

Each check reads the nodes through the API.

In [ ]:
chains = [call_node("GET", address, "/chain")[1] for address in NODES]                 # the chain of each node
pools = [call_node("GET", address, "/transactions/pending")[1] for address in NODES]   # the pending pool of each node
peers = [call_node("GET", address, "/nodes")[1] for address in NODES]                  # the peer list of each node
balances = {name: call_node("GET", NODE_C, f"/balance/{name}")[1]["confirmed"]         # the confirmed balance of each address,
            for name in ("alice", "bob", "carol", "miner-a", "miner-b")}               # read from node C
first = call_node("GET", NODE_C, f"/transactions/{txid}")[1]                           # Alice's payment, as node C sees it

checks = [                                                                             # each check is a description and a True or False
    ("every node holds 4 blocks", all(chain["length"] == 4 for chain in chains)),      # genesis plus three mined blocks
    ("every node ends on the same block", len({chain["chain"][-1]["hash"] for chain in chains}) == 1),   # one shared newest hash
    ("every block hash starts with four zeros", all(block["hash"].startswith("0000") for block in chains[0]["chain"])),   # proof of work
    ("every block points at the block before it", all(later["previous_hash"] == earlier["hash"] for earlier, later in zip(chains[0]["chain"], chains[0]["chain"][1:]))),   # the links
    ("every pending pool is empty", all(pool["count"] == 0 for pool in pools)),        # both payments were mined
    ("every node lists the other two as peers", all(peer["count"] == 2 for peer in peers)),   # a full mesh of three
    ("balances are Alice 29, Bob 14, Carol 5", (balances["alice"], balances["bob"], balances["carol"]) == (29, 14, 5)),   # value moved correctly
    ("each miner holds 51", balances["miner-a"] == 51 and balances["miner-b"] == 51),  # subsidy plus one fee each
    ("balances sum to 3 blocks x 50", sum(balances.values()) == 150),                  # nothing was created or lost
    ("Alice's payment has 2 confirmations on node C", first["status"] == "confirmed" and first["confirmations"] == 2),   # block 2 plus block 3
]
for description, passed in checks:                                                     # list every check
    print("PASS" if passed else "FAIL", description)                                   # with its outcome
print(f"\n{sum(passed for _, passed in checks)} passed, {sum(not passed for _, passed in checks)} failed")   # the final count

The three nodes keep running until the kernel is restarted, so the explorer stays available at http://127.0.0.1:5000/explorer. Running `stop_background_nodes()` in a new cell stops them sooner.